# Visualize One Edge-Flip Causal Replay

Replay the exact monochromatic-K5 participation changes caused by one edge flip. Each frame reveals one created or destroyed monochromatic K5 and updates the `(red, blue)` participation counts of its five vertices. The ordering is a visualization of simultaneous causal contributions, not physical or temporal propagation.

In [4]:
# Imports and Configuration

from pathlib import Path

import numpy as np
import plotly.graph_objects as go

from ramsey import (
    RGraph,
    RProblem,
    RSQLiteArchive,
    RSearchState,
)
from ramsey.REdgeFlipCausalAnalysis import (
    analyze_edge_flip_causality,
)

N_VERTICES = 43
ARCHIVE_ID = 1051
EDGE_INDEX = 767
FRAME_DURATION_MS = 900

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)

In [5]:
# Load the Archived Coloring and Analyze the Flip

graph = RGraph(
    RProblem.r55(
        n_vertices=N_VERTICES,
    )
)
archive = RSQLiteArchive(DATABASE_PATH)
archived = archive.load_coloring(
    ARCHIVE_ID,
    graph,
)
state = RSearchState(archived.coloring)
causal = analyze_edge_flip_causality(
    state,
    EDGE_INDEX,
)

# For replay only, group destroyed K5s before created K5s.
# This ordering is explicitly not a temporal claim.
events = tuple(sorted(
    causal.clique_changes,
    key=lambda change: (
        0 if change.destroyed else 1,
        change.clique_index,
    ),
))

print("Archive ID:", ARCHIVE_ID)
print("Score:", causal.score_before)
print("Edge:", causal.edge, causal.endpoints)
print("Color:", "red -> blue" if causal.old_color == 0 else "blue -> red")
print("Reward:", causal.exact_reward)
print("Causal K5 events:", len(events))
print()

for number, event in enumerate(events, start=1):
    action = "CREATE" if event.created else "DESTROY"
    color = "RED" if event.color == 0 else "BLUE"
    print(
        f"{number:2d}. {action:7s} {color:4s} "
        f"K5 {event.clique_index:6d}  "
        f"vertices={tuple(int(v) for v in event.vertices)}"
    )

Archive ID: 1051
Score: 350
Edge: 767 (26, 27)
Color: blue -> red
Reward: 5
Causal K5 events: 9

 1. DESTROY BLUE K5 462204  vertices=(5, 6, 9, 26, 27)
 2. DESTROY BLUE K5 466120  vertices=(5, 6, 20, 26, 27)
 3. DESTROY BLUE K5 518709  vertices=(5, 20, 26, 27, 31)
 4. DESTROY BLUE K5 544010  vertices=(6, 9, 26, 27, 32)
 5. DESTROY BLUE K5 563455  vertices=(6, 14, 20, 26, 27)
 6. DESTROY BLUE K5 577615  vertices=(6, 20, 26, 27, 32)
 7. DESTROY BLUE K5 856326  vertices=(14, 20, 26, 27, 31)
 8. CREATE  RED  K5 184690  vertices=(1, 12, 23, 26, 27)
 9. CREATE  RED  K5 210149  vertices=(1, 25, 26, 27, 37)


In [6]:
# Build the Interactive Causal Replay

angles = (
    np.pi / 2
    - 2 * np.pi * np.arange(N_VERTICES) / N_VERTICES
)
vertex_x = np.cos(angles)
vertex_y = np.sin(angles)
label_radius = 1.16
label_x = label_radius * vertex_x
label_y = label_radius * vertex_y

ACTIVITY_BANDS = 6

def edge_coordinates(edge_indices):
    x_values = []
    y_values = []

    for edge_index in edge_indices:
        u, v = graph.edges[int(edge_index)]
        u = int(u)
        v = int(v)
        x_values.extend((vertex_x[u], vertex_x[v], None))
        y_values.extend((vertex_y[u], vertex_y[v], None))

    return x_values, y_values

def participation_labels(participation):
    return [
        f"{vertex}<br>R{participation[vertex, 0]} B{participation[vertex, 1]}"
        for vertex in range(N_VERTICES)
    ]

def participation_hover(participation, hits):
    return [
        (
            f"Vertex {vertex}<br>"
            f"Red monochromatic K5s: {participation[vertex, 0]}<br>"
            f"Blue monochromatic K5s: {participation[vertex, 1]}<br>"
            f"Causal event hits: {hits[vertex]}"
        )
        for vertex in range(N_VERTICES)
    ]

def accumulated_edge_traces(edge_hits):
    traces = []

    for band in range(1, ACTIVITY_BANDS + 1):
        if band < ACTIVITY_BANDS:
            selected = np.flatnonzero(edge_hits == band)
        else:
            selected = np.flatnonzero(edge_hits >= band)

        x_values, y_values = edge_coordinates(selected)
        traces.append(
            go.Scatter(
                x=x_values,
                y=y_values,
                mode="lines",
                line=dict(
                    color="rgba(90, 90, 90, 0.40)",
                    width=0.7 + 0.65 * band,
                ),
                hoverinfo="skip",
                showlegend=False,
            )
        )

    return traces

def current_event_trace(event):
    if event is None:
        return go.Scatter(
            x=[], y=[], mode="lines",
            hoverinfo="skip", showlegend=False,
        )

    x_values, y_values = edge_coordinates(event.edges)
    color = (
        "rgba(220, 40, 40, 0.90)"
        if event.color == 0
        else "rgba(35, 90, 235, 0.90)"
    )

    return go.Scatter(
        x=x_values,
        y=y_values,
        mode="lines",
        line=dict(color=color, width=4.0),
        hoverinfo="skip",
        showlegend=False,
    )

def flipped_edge_trace():
    x_values, y_values = edge_coordinates([causal.edge])
    return go.Scatter(
        x=x_values,
        y=y_values,
        mode="lines",
        line=dict(color="#f4b400", width=6.0),
        hoverinfo="skip",
        showlegend=False,
    )

def vertex_traces(participation, vertex_hits, current_vertices):
    sizes = 11 + 2.6 * np.sqrt(vertex_hits)
    maximum_hits = max(1, int(vertex_hits.max()))

    nodes = go.Scatter(
        x=vertex_x,
        y=vertex_y,
        mode="markers",
        marker=dict(
            size=sizes,
            color=vertex_hits,
            colorscale="Viridis",
            cmin=0,
            cmax=maximum_hits,
            line=dict(color="#202020", width=1),
            showscale=False,
        ),
        hovertext=participation_hover(participation, vertex_hits),
        hoverinfo="text",
        showlegend=False,
    )

    labels = go.Scatter(
        x=label_x,
        y=label_y,
        mode="text",
        text=participation_labels(participation),
        textfont=dict(size=10, color="#182a4a"),
        hoverinfo="skip",
        showlegend=False,
    )

    current_vertices = np.asarray(current_vertices, dtype=np.int32)
    current = go.Scatter(
        x=vertex_x[current_vertices] if len(current_vertices) else [],
        y=vertex_y[current_vertices] if len(current_vertices) else [],
        mode="markers",
        marker=dict(
            size=22,
            symbol="circle-open",
            color="#f4b400",
            line=dict(width=3),
        ),
        hoverinfo="skip",
        showlegend=False,
    )

    return [nodes, labels, current]

def frame_data(participation, vertex_hits, edge_hits, event):
    current_vertices = (
        event.vertices if event is not None else []
    )
    return [
        *accumulated_edge_traces(edge_hits),
        current_event_trace(event),
        flipped_edge_trace(),
        *vertex_traces(
            participation,
            vertex_hits,
            current_vertices,
        ),
    ]

participation = causal.participation_before.vertices.copy()
vertex_hits = np.zeros(N_VERTICES, dtype=np.int32)
edge_hits = np.zeros(graph.number_of_edges, dtype=np.int32)

initial_data = frame_data(
    participation,
    vertex_hits,
    edge_hits,
    None,
)

frames = [
    go.Frame(
        name="0",
        data=initial_data,
        layout=go.Layout(
            title_text=(
                f"Archive {ARCHIVE_ID}, score {causal.score_before} — "
                f"flip edge {causal.edge} {causal.endpoints} — start"
            )
        ),
    )
]

for event_number, event in enumerate(events, start=1):
    participation[
        event.vertices,
        event.color,
    ] += event.delta
    vertex_hits[event.vertices] += 1
    edge_hits[event.edges] += 1

    action = "create" if event.created else "destroy"
    color = "red" if event.color == 0 else "blue"

    frames.append(
        go.Frame(
            name=str(event_number),
            data=frame_data(
                participation.copy(),
                vertex_hits.copy(),
                edge_hits.copy(),
                event,
            ),
            layout=go.Layout(
                title_text=(
                    f"Causal replay {event_number}/{len(events)} — "
                    f"{action} {color} K5 — vertices "
                    f"{tuple(int(v) for v in event.vertices)}"
                )
            ),
        )
    )

if not np.array_equal(
    participation,
    causal.participation_after.vertices,
):
    raise RuntimeError(
        "Causal replay did not reconstruct final vertex participation."
    )

figure = go.Figure(
    data=initial_data,
    frames=frames,
)

slider_steps = [
    dict(
        method="animate",
        label=str(number),
        args=[
            [str(number)],
            dict(
                mode="immediate",
                frame=dict(duration=0, redraw=True),
                transition=dict(duration=0),
            ),
        ],
    )
    for number in range(0, len(events) + 1)
]

figure.update_layout(
    title=(
        f"Archive {ARCHIVE_ID}, score {causal.score_before} — "
        f"flip edge {causal.edge} {causal.endpoints} — start"
    ),
    autosize=True,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=90, b=80),
    xaxis=dict(
        visible=False,
        range=[-1.32, 1.32],
        scaleanchor="y",
        scaleratio=1,
    ),
    yaxis=dict(
        visible=False,
        range=[-1.32, 1.32],
    ),
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.02,
            y=-0.03,
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[
                        None,
                        dict(
                            fromcurrent=True,
                            frame=dict(
                                duration=FRAME_DURATION_MS,
                                redraw=True,
                            ),
                            transition=dict(duration=150),
                        ),
                    ],
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[
                        [None],
                        dict(
                            mode="immediate",
                            frame=dict(duration=0, redraw=False),
                            transition=dict(duration=0),
                        ),
                    ],
                ),
            ],
        )
    ],
    sliders=[
        dict(
            active=0,
            x=0.18,
            len=0.78,
            currentvalue=dict(prefix="Causal event: "),
            steps=slider_steps,
        )
    ],
)

figure.show(
    renderer="browser",
    config=dict(
        responsive=True,
        scrollZoom=True,
        displaylogo=False,
    ),
)